# Setup

In [5]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error as mse

import matplotlib.pyplot as plt
import seaborn as sns

# Start with the default (light) style
plt.style.use('default')

# Now update the rcParams to match your desired white-background style
plt.rcParams.update({
    'axes.facecolor': 'white',         # axes background
    'figure.facecolor': 'white',       # figure background
    'axes.edgecolor': 'black',         # border of plot
    'axes.labelcolor': 'black',        # axis labels
    'xtick.color': 'black',
    'ytick.color': 'black',
    'grid.color': 'black',             # grid lines color
    'grid.linewidth': 0.5,
    'grid.alpha': 0.2,                 # faint grid
    'font.size': 12,
    'legend.frameon': True,
    'lines.linewidth': 2,
    'figure.dpi': 150
})

def variable_df(dataframe, variable, model, datetime_component, norm=False, quantiles=[0.1, 0.25, 0.5, 0.75, 0.9]):
    # Step 1: Round and convert

    if variable == 'cost' and datetime_component == 'month':
        dataframe = dataframe.resample('D').sum()

    df_rounded = dataframe.round(4).astype('float32')

    # Step 2: Slice by variable
    df_var = df_rounded.xs(variable, level='variable', axis=1)

    # Step 3: Normalize if needed
    if norm:
        df_var = df_var / (df_var.iloc[0] * 2)

    # Step 4: Slice by model
    df_model = df_var.xs(model, level='dictionary', axis=1)

    # Step 5: Create datetime component column (e.g., hour)
    df_model = df_model.copy()
    df_model[datetime_component] = getattr(df_model.index, datetime_component)

    # Step 6: Melt to long format (one value per row)
    df_melted = df_model.melt(id_vars=datetime_component, var_name='building', value_name='value')

    # Step 7: Group by datetime component (e.g., hour) and compute quantiles across buildings
    df_quantiles = df_melted.groupby(datetime_component)['value'].quantile(quantiles).unstack()
    df_quantiles.columns = [f'Q{int(q*100):02d}' for q in quantiles]

    return df_quantiles

def plot_scatter_plot(mse_data, cost_data):
    # Create a figure
    plt.figure(figsize=(8, 5), dpi=180)

    # Scatter plot for each model
    for model in mse_data.keys():
        # Get the MSE and cost values for each building
        mse_values = mse_data[model]
        cost_values = cost_data[model]

        # Scatter plot
        plt.scatter(mse_values, cost_values, label=model, alpha=0.7)

    # Set labels and title
    plt.xlabel("RMSE", fontsize=12)
    plt.ylabel("Cost", fontsize=12)
    plt.title("Scatter Plot of Cost vs. RMSE", fontsize=18, fontweight="bold")

    # Add a legend
    plt.legend(title="Models", loc="lower left", fontsize=14)

    # Show grid for better visibility
    plt.grid(True, linestyle="--", alpha=0.6)

    # Show the plot
    plt.tight_layout()
    plt.show()

def hourly_quantile_plot(stats, size=(14, 6), title='Hourly Distribution Across Buildings and Days'):
    plt.figure(figsize=size)

    # Plot median
    plt.plot(stats.index, stats['Q50'], label='Median (Q50)', color='blue')

    # Fill interquartile range (Q25–Q75)
    if 'Q25' in stats.columns and 'Q75' in stats.columns:
        plt.fill_between(
            stats.index,
            stats['Q25'],
            stats['Q75'],
            color='blue',
            alpha=0.25,
            label='Interquartile Range (Q25–Q75)'
        )

    # Fill outer range (Q10–Q90)
    if 'Q10' in stats.columns and 'Q90' in stats.columns:
        plt.fill_between(
            stats.index,
            stats['Q10'],
            stats['Q90'],
            color='blue',
            alpha=0.15,
            label='10th–90th Percentile Range'
        )

    plt.xticks(range(len(stats)))
    plt.xlabel('Hour of Day')
    plt.ylabel('Value')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def compare_hourly_quantile_plots(stats_list, rowtitles, columns=3, rows=1, titles=None, x_axis='Hour of the day', size=(18, 5)):
    n = len(stats_list)
    if titles is None:
        titles = [f"Plot {i+1}" for i in range(n)]

    # Step 1: Find global min and max across all stats
    global_min = min(df[[col for col in df.columns if col.startswith('Q')]].min().min() for df in stats_list)
    global_max = max(df[[col for col in df.columns if col.startswith('Q')]].max().max() for df in stats_list)

    # Step 2: Create subplots (note: rows first!)
    fig, axes = plt.subplots(rows, columns, figsize=size, sharey=True)

    # Flatten axes for easy iteration
    axes = axes.flatten() if isinstance(axes, (list, np.ndarray)) else [axes]

    for row in range(rows):
            # Compute y position (between 0 and 1, from bottom to top)
            y_pos = 1 - (row + 0.5) / rows
            fig.text(0, y_pos, rowtitles[row], ha='center', va='center',rotation=90, fontsize=14, fontweight='bold')

    # Step 3: Plot each
    for i, (ax, stats, title) in enumerate(zip(axes, stats_list, titles)):
        line_color = '#66B2FF'
        fill_iqr_color = '#66B2FF'
        fill_outer_color = '#66B2FF'

        ax.plot(stats.index, stats['Q50'], label='Median (Q50)', color=line_color)

        if 'Q25' in stats.columns and 'Q75' in stats.columns:
            ax.fill_between(
                stats.index, stats['Q25'], stats['Q75'],
                color=fill_iqr_color, alpha=0.4,
                label='IQR (Q25–Q75)' if i == 0 else None
            )

        if 'Q10' in stats.columns and 'Q90' in stats.columns:
            ax.fill_between(
                stats.index, stats['Q10'], stats['Q90'],
                color=fill_outer_color, alpha=0.2,
                label='P10–P90' if i == 0 else None
            )

        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)

        ax.set_title(title)
        ax.set_xlabel(x_axis)
        ax.grid(True)
        ax.set_xticks(range(0, len(stats), 2))
        ax.set_ylim(global_min, global_max * 1.1)


    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3)

    fig.suptitle('Averaged hourly PV forecasts per model', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.show()

def hourly_combo_confidence_plot(stats1, stats2, label1='Dataset 1', label2='Dataset 2', color1='blue', color2='green'):

    # Plotting
    plt.figure(figsize=(14, 6))

    # Plot mean lines
    plt.plot(stats1.index, stats1['mean'], label=f'{label1} Mean', color=color1)
    plt.plot(stats2.index, stats2['mean'], label=f'{label2} Mean', color=color2)

    # Plot confidence intervals
    for i, alpha in zip([1, 2], [0.3, 0.15]):
        plt.fill_between(
            stats1.index,
            stats1['mean'] - i * stats1['std'],
            stats1['mean'] + i * stats1['std'],
            color=color1,
            alpha=alpha,
            label=f'{label1} ±{i} Std Dev' if i == 1 else None  # Avoid repeating in legend
        )
        plt.fill_between(
            stats2.index,
            stats2['mean'] - i * stats2['std'],
            stats2['mean'] + i * stats2['std'],
            color=color2,
            alpha=alpha,
            label=f'{label2} ±{i} Std Dev' if i == 1 else None
        )

    plt.xticks(range(len(stats1)))
    plt.ylim(min(min(stats1['mean'] - 3 * stats1['std']),min(stats2['mean'] - 3 * stats2['std'])), max(max(stats1['mean'] + 3 * stats1['std']),max(stats2['mean'] + 3 * stats2['std'])))
    plt.xlabel('Hour of Day')
    plt.ylabel('Value')
    plt.title('Hourly Average with Confidence Intervals for Two Datasets')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [6]:
noise = 0.5

In [7]:
df = pd.read_pickle("../results/all_data_" + str(noise) + "_noise.pkl")
df_long = pd.read_pickle("../results/all_data_long_" + str(noise) + "_noise.pkl")

In [8]:
df_long = df_long.set_index('datetime')

In [9]:
buildings = list(df_long.building.unique())
models = list(df_long.dictionary.unique())

In [10]:
model_mse = {model: [] for model in models}
model_costs = {model: [] for model in models}

for model in models:
    for building in buildings:
        model_mse[model].append(mse(df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'pv')].value /               # True values divided by
                                    max(df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'pv')].value),           # Maximum of true values
                                    df_long[(df_long['building'] == building) & (df_long['dictionary'] == model) & (df_long['variable'] == 'pv')].value /                   # Predicted values divided by
                                    max(df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'pv')].value),           # Maximum of true values
                                    squared=False))                                                                                                                         # Root of the MSE

        model_costs[model].append(sum(df_long[(df_long['building'] == building) & (df_long['dictionary'] == model) & (df_long['variable'] == 'cost')].value)) # profit for injection

In [11]:
no_opt_costs = []
for building in buildings:
    # Calculate net energy: positive = surplus, negative = deficit
    net_energy = np.array((df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'pv')].value -
                             df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'load')].value))
    no_opt_costs.append(
        sum(
            np.where(
                     net_energy < 0,
                     -net_energy * np.array(df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'offtake')].value),      # Buying energy: cost
                     -net_energy * np.array(df_long[(df_long['building'] == building) & (df_long['dictionary'] == 'perfect') & (df_long['variable'] == 'injection')].value)    # Selling energy: revenue (negative cost)
        )))

In [12]:
for key in model_mse.keys():
    print(np.average(model_mse[key]))

0.19900651496682537
0.08228927636033215
0.13699263841798878
0.13790586564785182
0.0


In [15]:
for key in model_mse.keys():
    print(key + ': ' + str(np.average(model_costs[key])))
print(np.average(no_opt_costs))

cvx: 368.7422906406108
lstm: 376.5949815098905
lstm_cvx: 358.6625461417177
naive: 393.06598002717135
perfect: 281.3504441331464
499.82060113472124


In [ ]:
cost_df = pd.DataFrame(model_costs, index=buildings)
cost_df.rename(columns={
    'cvx': 'VOF',
    'lstm_cvx': 'VOF-WS'
}, inplace=True)

In [ ]:
rmse_df = pd.DataFrame(model_mse, index=buildings)
rmse_df.rename(columns={
    'cvx': 'VOF',
    'lstm_cvx': 'VOF-WS'
}, inplace=True)

In [ ]:
cost_df_norm = cost_df.copy()
columns_to_normalize = cost_df.columns

cost_df_norm[columns_to_normalize] = cost_df[columns_to_normalize].apply(
    lambda row: (row - row.min()) / (row.max() - row.min()), axis=1
)

rmse_df_norm = rmse_df.copy()

rmse_df_norm[columns_to_normalize] = rmse_df[columns_to_normalize].apply(
    lambda row: (row - row.min()) / (row.max() - row.min()), axis=1
)

In [25]:
# Call the function to plot the scatter plot
plot_scatter_plot(rmse_df_norm[rmse_df_norm.columns[:-1]], cost_df_norm[cost_df_norm.columns[:-1]])

NameError: name 'rmse_df_norm' is not defined

In [ ]:
var = 'pv'
time = 'hour'

In [ ]:
df_winter = df[df.index.month.isin([11, 12, 1, 2])]
df_spring = df[df.index.month.isin([3, 4, 9, 10])]
df_summer = df[df.index.month.isin([5, 6, 7, 8])]

In [ ]:
zzzdf_lstm_winter = variable_df(df_winter, var,'lstm', time)
df_lstm_cvx_winter = variable_df(df_winter, var,'lstm_cvx', time)
df_cvx_winter = variable_df(df_winter, var,'cvx', time)
df_perfect_winter = variable_df(df_winter, var,'perfect', time)

df_lstm_spring = variable_df(df_spring, var,'lstm', time)
df_lstm_cvx_spring = variable_df(df_spring, var,'lstm_cvx', time)
df_cvx_spring = variable_df(df_spring, var,'cvx', time)
df_perfect_spring = variable_df(df_spring, var,'perfect', time)

df_lstm_summer = variable_df(df_summer, var,'lstm', time)
df_lstm_cvx_summer = variable_df(df_summer, var,'lstm_cvx', time)
df_cvx_summer = variable_df(df_summer, var,'cvx', time)
df_perfect_summer = variable_df(df_summer, var,'perfect', time)

In [ ]:
 compare_hourly_quantile_plots([df_perfect_winter, df_lstm_winter, df_lstm_cvx_winter, df_cvx_winter, df_perfect_spring, df_lstm_spring, df_lstm_cvx_spring, df_cvx_spring, df_perfect_summer, df_lstm_summer, df_lstm_cvx_summer, df_cvx_summer], ['Winter', 'Spring and Autumn', 'Summer'], columns=4, rows=3, titles=['Actual', 'LSTM', 'VOF-WS', 'VOF', '', '', '', '','', '', '', ''], size=(18,8), x_axis='Hour of the day')

In [ ]:
time = 'hour'

In [ ]:
var = 'charge'
charge_lstm_winter = variable_df(df_winter, var,'lstm', time, quantiles=[0.5])
charge_lstm_spring = variable_df(df_spring, var,'lstm', time, quantiles=[0.5])
charge_lstm_summer = variable_df(df_summer, var,'lstm', time, quantiles=[0.5])
charge_lstm_cvx_winter = variable_df(df_winter, var,'lstm_cvx', time, quantiles=[0.5])
charge_lstm_cvx_spring = variable_df(df_spring, var,'lstm_cvx', time, quantiles=[0.5])
charge_lstm_cvx_summer = variable_df(df_summer, var,'lstm_cvx', time, quantiles=[0.5])
charge_perfect_winter = variable_df(df_winter, var,'perfect', time, quantiles=[0.5])
charge_perfect_spring = variable_df(df_spring, var,'perfect', time, quantiles=[0.5])
charge_perfect_summer = variable_df(df_summer, var,'perfect', time, quantiles=[0.5])

var = 'discharge'
discharge_lstm_winter = variable_df(df_winter, var,'lstm', time, quantiles=[0.5])
discharge_lstm_spring = variable_df(df_spring, var,'lstm', time, quantiles=[0.5])
discharge_lstm_summer = variable_df(df, var,'lstm', time, quantiles=[0.5])
discharge_lstm_cvx_winter = variable_df(df_winter, var,'lstm_cvx', time, quantiles=[0.5])
discharge_lstm_cvx_spring = variable_df(df_spring, var,'lstm_cvx', time, quantiles=[0.5])
discharge_lstm_cvx_summer = variable_df(df, var,'lstm_cvx', time, quantiles=[0.5])
discharge_perfect_winter = variable_df(df_winter, var,'perfect', time, quantiles=[0.5])
discharge_perfect_spring = variable_df(df_spring, var,'perfect', time, quantiles=[0.5])
discharge_perfect_summer = variable_df(df_summer, var,'perfect', time, quantiles=[0.5])
# Prepare x-axis
common_index = charge_lstm_winter.index
x = np.arange(len(common_index))
width = 0.4  # width of each bar
overlap = 0.3  # amount of overlap between bars

fig, (ax1, ax2, ax3) = plt.subplots(nrows=3, figsize=(15, 8), sharex=True)

# === TOP: LSTM vs. LSTM-CVX ===
# Charge
ax1.bar(x - (width - overlap)/2, charge_lstm_winter['Q50'], width, label='Charge LSTM', color='r')
ax1.bar(x + (width - overlap)/2, charge_lstm_cvx_winter['Q50'], width, label='Charge LSTM-CVX', color='g')
# Discharge (as negative)
ax1.bar(x - (width - overlap)/2, -discharge_lstm_winter['Q50'], width, label='Discharge LSTM', color='r', alpha=0.4, hatch='//')
ax1.bar(x + (width - overlap)/2, -discharge_lstm_cvx_winter['Q50'], width, label='Discharge LSTM-CVX', color='g', alpha=0.4, hatch='//')

ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_ylabel('Energy')
ax1.set_title('Winter: Charge & Discharge')
ax1.legend(loc='upper left', fontsize=10, ncol=2)

# === MIDDLE: CVX vs. Perfect ===
# Charge
ax2.bar(x - (width - overlap)/2, charge_lstm_spring['Q50'], width, label='Charge LSTM', color='r')
ax2.bar(x + (width - overlap)/2, charge_lstm_cvx_spring['Q50'], width, label='Charge LSTM-CVX', color='g')
# Discharge (as negative)
ax2.bar(x - (width - overlap)/2, -discharge_lstm_spring['Q50'], width, label='Discharge LSTM', color='r', alpha=0.4, hatch='//')
ax2.bar(x + (width - overlap)/2, -discharge_lstm_cvx_spring['Q50'], width, label='Discharge LSTM-CVX', color='g', alpha=0.4, hatch='//')

ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Energy')
ax2.set_title('Spring & Autumn: Charge & Discharge')
ax2.legend(loc='upper left', fontsize=10, ncol=2)

# === BOTTOM: CVX vs. Perfect ===
# Charge
ax3.bar(x - (width - overlap)/2, charge_lstm_summer['Q50'], width, label='Charge LSTM', color='r')
ax3.bar(x + (width - overlap)/2, charge_lstm_cvx_summer['Q50'], width, label='Charge LSTM-CVX', color='g')
# Discharge (as negative)
ax3.bar(x - (width - overlap)/2, -discharge_lstm_summer['Q50'], width, label='Discharge LSTM', color='r', alpha=0.4, hatch='//')
ax3.bar(x + (width - overlap)/2, -discharge_lstm_cvx_summer['Q50'], width, label='Discharge LSTM-CVX', color='g', alpha=0.4, hatch='//')

ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_ylabel('Energy')
ax3.set_title('Summer: Charge & Discharge')
ax3.set_xlabel('Hour of the day')
ax3.set_xticks(x)
ax3.set_xticklabels(common_index, rotation=45)
ax3.legend(loc='upper left', fontsize=10, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 6), sharex=True)
axes = axes.flatten()  # Flatten to make indexing easy
width = 0.4  # width of each bar
overlap = 0.15  # amount of overlap between bars
# Axes assignment
ax1, ax2, ax3, ax4, ax5, ax6 = axes
common_index = charge_lstm_winter.index
x = np.arange(len(common_index))
# === TOP: LSTM vs. LSTM-CVX ===
# Charge
ax1.bar(x - (width - overlap)/2, charge_lstm_winter['Q50'] - charge_perfect_winter['Q50'], width, label='LSTM vs real', color='r')
ax1.bar(x + (width - overlap)/2, charge_lstm_cvx_winter['Q50'] - charge_perfect_winter['Q50'], width, label='VOF-WS vs real', color='g')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_xticks(range(0, len(common_index), 2))
ax1.set_ylabel('Difference in charging')
ax1.set_title('Winter')
ax1.legend(loc='upper left', fontsize=10)

# Discharge
ax4.bar(x - (width - overlap)/2, -discharge_lstm_winter['Q50'] - (-discharge_perfect_winter['Q50']), width, label='LSTM vs real', color='r', alpha=0.4, hatch='//')
ax4.bar(x + (width - overlap)/2, -discharge_lstm_cvx_winter['Q50'] - (-discharge_perfect_winter['Q50']), width, label='VOF-WS vs real', color='g', alpha=0.4, hatch='//')
ax4.axhline(0, color='black', linewidth=0.8)
ax4.set_ylabel('Difference in discharging')
ax4.set_xticks(range(0, len(common_index), 2))
ax4.set_xlabel('Hour of the day')
ax4.legend(loc='upper left', fontsize=10)

# === MIDDLE: Spring & Autumn ===
# Charge
ax2.bar(x - (width - overlap)/2, charge_lstm_spring['Q50'] - charge_perfect_spring['Q50'], width, label='LSTM vs real', color='r')
ax2.bar(x + (width - overlap)/2, charge_lstm_cvx_spring['Q50'] - charge_perfect_spring['Q50'], width, label='VOF-WS vs real', color='g')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(range(0, len(common_index), 2))
ax2.set_title('Spring & Autumn')
ax2.legend(loc='upper left', fontsize=10)

# Discharge
ax5.bar(x - (width - overlap)/2, -discharge_lstm_spring['Q50'] - (-discharge_perfect_spring['Q50']), width, label='LSTM vs real', color='r', alpha=0.4, hatch='//')
ax5.bar(x + (width - overlap)/2, -discharge_lstm_cvx_spring['Q50'] - (-discharge_perfect_spring['Q50']), width, label='VOF-WS vs real', color='g', alpha=0.4, hatch='//')
ax5.axhline(0, color='black', linewidth=0.8)
ax5.set_xlabel('Hour of the day')
ax5.set_xticks(range(0, len(common_index), 2))
ax5.legend(loc='upper left', fontsize=10)

# === BOTTOM: Summer ===
# Charge
ax3.bar(x - (width - overlap)/2, charge_lstm_summer['Q50'] - charge_perfect_summer['Q50'], width, label='LSTM vs real', color='r')
ax3.bar(x + (width - overlap)/2, charge_lstm_cvx_summer['Q50'] - charge_perfect_summer['Q50'], width, label='VOF-WS vs real', color='g')
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xticks(range(0, len(common_index), 2))
ax3.set_title('Summer')
ax3.legend(loc='upper left', fontsize=10)

# Discharge
ax6.bar(x - (width - overlap)/2, -discharge_lstm_summer['Q50'] - (-discharge_perfect_summer['Q50']), width, label='LSTM vs real', color='r', alpha=0.4, hatch='//')
ax6.bar(x + (width - overlap)/2, -discharge_lstm_cvx_summer['Q50'] - (-discharge_perfect_summer['Q50']), width, label='VOF-WS vs real', color='g', alpha=0.4, hatch='//')
ax6.axhline(0, color='black', linewidth=0.8)
ax6.set_xlabel('Hour of the day')
ax6.set_xticks(range(0, len(common_index), 2))
ax6.legend(loc='upper left', fontsize=10)
# Set consistent y-axis limits for charge plots (ax1, ax2, ax3)
charge_axes = [ax1, ax2, ax3]
charge_ys = []

for ax in charge_axes:
    for bar in ax.patches:
        charge_ys.append(bar.get_height())

charge_ymin = min(charge_ys)
charge_ymax = max(charge_ys)

for ax in charge_axes:
    ax.set_ylim(charge_ymin*1.1, charge_ymax*1.1)

# Set consistent y-axis limits for discharge plots (ax4, ax5, ax6)
discharge_axes = [ax4, ax5, ax6]
discharge_ys = []

for ax in discharge_axes:
    for bar in ax.patches:
        discharge_ys.append(bar.get_height())

discharge_ymin = min(discharge_ys)
discharge_ymax = max(discharge_ys)

for ax in discharge_axes:
    ax.set_ylim(discharge_ymin*1.1, discharge_ymax*1.1)
plt.tight_layout()
plt.show()


In [ ]:
df_winter_perfect = df[df.index.month.isin([11, 12, 1, 2])].xs('cost', level='variable', axis=1).xs('perfect', level='dictionary', axis=1)
df_spring_perfect = df[df.index.month.isin([3, 4, 9, 10])].xs('cost', level='variable', axis=1).xs('perfect', level='dictionary', axis=1)
df_summer_perfect = df[df.index.month.isin([5, 6, 7, 8])].xs('cost', level='variable', axis=1).xs('perfect', level='dictionary', axis=1)

df_winter_lstm = df[df.index.month.isin([11, 12, 1, 2])].xs('cost', level='variable', axis=1).xs('lstm', level='dictionary', axis=1)
df_spring_lstm = df[df.index.month.isin([3, 4, 9, 10])].xs('cost', level='variable', axis=1).xs('lstm', level='dictionary', axis=1)
df_summer_lstm = df[df.index.month.isin([5, 6, 7, 8])].xs('cost', level='variable', axis=1).xs('lstm', level='dictionary', axis=1)

df_winter_lstmcvx = df[df.index.month.isin([11, 12, 1, 2])].xs('cost', level='variable', axis=1).xs('lstm_cvx', level='dictionary', axis=1)
df_spring_lstmcvx = df[df.index.month.isin([3, 4, 9, 10])].xs('cost', level='variable', axis=1).xs('lstm_cvx', level='dictionary', axis=1)
df_summer_lstmcvx = df[df.index.month.isin([5, 6, 7, 8])].xs('cost', level='variable', axis=1).xs('lstm_cvx', level='dictionary', axis=1)

In [ ]:
cost_array = np.array([[sum(sum(df_winter_perfect.values)), sum(sum(df_winter_lstm.values)), sum(sum(df_winter_lstmcvx.values))],
                        [sum(sum(df_spring_perfect.values)), sum(sum(df_spring_lstm.values)), sum(sum(df_spring_lstmcvx.values))],
                        [sum(sum(df_summer_perfect.values)), sum(sum(df_summer_lstm.values)), sum(sum(df_summer_lstmcvx.values))]])

In [ ]:
cost_array_scaled = (cost_array / cost_array[:, [0]])

In [ ]:
cost_array_scaled - cost_array_scaled[:, [-1]]